# Investigação do pré-processamento

Este notebook documenta como foram definidas as decisões de pré-processamento para o futuro estudo de classificação. Ele examina o recorte de seis classes, as duplicatas, a representação de `FINGER_POSITIONS` e as ambiguidades dos atributos escolhidos.

O subconjunto categórico é exportado para consultas futuras. O notebook de experimentos refaz o pré-processamento a partir do CSV original.


In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "chord-fingers.csv").exists():
    ROOT = ROOT.parent
SOURCE = ROOT / "data" / "raw" / "chord-fingers.csv"
if not SOURCE.exists():
    raise FileNotFoundError("Execute o notebook na raiz do projeto ou em notebooks/.")

CLASSES = ("maj", "m", "dim", "7", "maj7", "m7")
FINGER_COLUMNS = tuple(f"STRING_{position}" for position in range(1, 7))
ALLOWED_FINGERS = {"x", "0", "1", "2", "3", "4"}


## 1. Seleção das classes

O alvo continua sendo `CHORD_TYPE`. O recorte considera exclusivamente `maj`, `m`, `dim`, `7`, `maj7` e `m7`.


In [2]:
raw = pd.read_csv(SOURCE, sep=";", dtype="string", keep_default_na=False)
selected = raw.loc[raw["CHORD_TYPE"].isin(CLASSES)].copy()
assert len(selected) == 965, f"Esperados 965 registros; encontrados {len(selected)}"

class_distribution = (
    selected["CHORD_TYPE"]
    .value_counts()
    .reindex(CLASSES)
    .rename("records")
    .to_frame()
)
class_distribution["percentage"] = (100 * class_distribution["records"] / len(selected)).round(2)
class_distribution


,records,percentage
CHORD_TYPE,,
maj,203,21.04
m,154,15.96
dim,86,8.91
7,208,21.55
maj7,137,14.2
m7,177,18.34


## 2. Duplicatas exatas

A comparação usa todas as colunas originais. Isso evita considerar como duplicados dois registros que parecem iguais apenas depois da retirada de `CHORD_STRUCTURE` e `NOTE_NAMES`.


In [3]:
duplicate_mask = selected.duplicated(keep=False)
duplicate_rows = selected.loc[duplicate_mask].sort_values(list(selected.columns))
deduplicated = selected.drop_duplicates().copy()
removed_duplicates = len(selected) - len(deduplicated)

print(f"Registros antes da remoção: {len(selected)}")
print(f"Duplicatas exatas removidas: {removed_duplicates}")
print(f"Registros restantes: {len(deduplicated)}")
duplicate_rows


Registros antes da remoção: 965
Duplicatas exatas removidas: 6
Registros restantes: 959


,CHORD_ROOT,CHORD_TYPE,CHORD_STRUCTURE,FINGER_POSITIONS,NOTE_NAMES
1436,B,7,1;3;5;b7,"x,2,1,3,0,4","B,D#,A,B,F#"
1442,B,7,1;3;5;b7,"x,2,1,3,0,4","B,D#,A,B,F#"
1485,B,m,1;b3;5,"x,1,3,4,2,1","B,F#,B,D,F#"
1491,B,m,1;b3;5,"x,1,3,4,2,1","B,F#,B,D,F#"
1612,C,7,1;3;5;b7,"x,3,2,4,1,x","C,E,Bb,C"
1615,C,7,1;3;5;b7,"x,3,2,4,1,x","C,E,Bb,C"
1720,C,m7,1;b3;5;b7,"x,2,1,3,4,x","C,Eb,Bb,Eb"
1721,C,m7,1;b3;5;b7,"x,2,1,3,4,x","C,Eb,Bb,Eb"
1760,C,maj7,1;3;5;7,"x,x,2,4,1,3","E,B,C,G"
1762,C,maj7,1;3;5;7,"x,x,2,4,1,3","E,B,C,G"


## 3. Representação dos atributos

`CHORD_STRUCTURE` e `NOTE_NAMES` revelam a composição harmônica e não serão usados como preditores. `CHORD_ROOT` permanece categórico. `FINGER_POSITIONS` contém seis símbolos separados por vírgula, na ordem da corda E grave à mais aguda. Os valores `1` a `4` indicam dedos, não casas do instrumento.


In [4]:
finger_parts = deduplicated["FINGER_POSITIONS"].str.split(",")
valid_positions = finger_parts.map(
    lambda values: len(values) == 6 and all(value in ALLOWED_FINGERS for value in values)
)
assert valid_positions.all(), "Foram encontradas posições de dedos fora do formato esperado."

finger_features = pd.DataFrame(
    finger_parts.tolist(),
    index=deduplicated.index,
    columns=FINGER_COLUMNS,
    dtype="string",
)
candidate_data = pd.concat(
    [deduplicated[["CHORD_ROOT"]], finger_features, deduplicated[["CHORD_TYPE"]]],
    axis=1,
).reset_index(drop=True)

print("Dimensões candidatas (registros, preditores + alvo):", candidate_data.shape)
candidate_data.head()


Dimensões candidatas (registros, preditores + alvo): (959, 8)


,CHORD_ROOT,STRING_1,STRING_2,STRING_3,STRING_4,STRING_5,STRING_6,CHORD_TYPE
0,A#,x,1,3,1,4,1,7
1,A#,1,3,1,2,4,1,7
2,A#,x,x,1,3,2,4,7
3,A#,x,1,3,3,3,x,maj
4,A#,1,3,4,2,1,1,maj


## 4. Ambiguidades dos atributos escolhidos

Uma combinação é ambígua quando a mesma raiz e a mesma sequência de dedos aparecem associadas a mais de uma classe. Esses casos revelam uma limitação dos preditores e não devem ser descartados automaticamente.


In [5]:
groups = deduplicated.groupby(["CHORD_ROOT", "FINGER_POSITIONS"], dropna=False)["CHORD_TYPE"]
combinations = groups.agg(
    record_count="size",
    class_count="nunique",
    classes=lambda values: " | ".join(sorted(values.unique())),
).reset_index()
ambiguous = combinations.loc[combinations["class_count"] > 1].copy()

ambiguous_keys = ambiguous.set_index(["CHORD_ROOT", "FINGER_POSITIONS"]).index
record_keys = deduplicated.set_index(["CHORD_ROOT", "FINGER_POSITIONS"]).index
affected_labels = deduplicated.loc[record_keys.isin(ambiguous_keys), "CHORD_TYPE"]
ambiguity_by_class = affected_labels.value_counts().reindex(CLASSES, fill_value=0)

print(f"Combinações distintas: {len(combinations)}")
print(f"Combinações ambíguas: {len(ambiguous)}")
print(f"Registros afetados: {int(ambiguous['record_count'].sum())}")
display(ambiguity_by_class.rename("affected_records").to_frame())
ambiguous


Combinações distintas: 927
Combinações ambíguas: 31
Registros afetados: 63


,affected_records
CHORD_TYPE,
maj,10
m,10
dim,9
7,16
maj7,8
m7,10


,CHORD_ROOT,FINGER_POSITIONS,record_count,class_count,classes
28,A,"x,0,1,3,4,2",2,2,m | maj
59,A,"x,2,1,3,x,x",2,2,7 | m7
79,A,"x,x,x,2,1,0",3,2,m | maj
111,Ab,"1,x,2,3,x,x",2,2,7 | maj7
202,B,"x,x,1,2,4,3",2,2,dim | maj
211,B,"x,x,x,3,2,1",2,2,dim | m
255,Bb,"x,x,1,1,1,4",2,2,maj | maj7
273,Bb,"x,x,x,2,1,3",2,2,7 | dim
275,Bb,"x,x,x,3,2,1",2,2,7 | m
335,C,"x,x,1,1,1,3",2,2,7 | maj7


## 5. Exportação para consulta


In [ ]:
output_path = ROOT / "data" / "processed" / "chords_six_classes.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
candidate_data.to_csv(output_path, index=False, encoding="utf-8")
print(f"CSV gerado em {output_path.relative_to(ROOT)}")


## 6. Decisões adotadas nos experimentos

A investigação fundamentou as seguintes decisões implementadas em `experiments.ipynb`:

1. selecionar as seis classes e remover as 6 duplicatas exatas, resultando em 959 registros;
2. usar `CHORD_ROOT` e as seis cordas como sete preditores categóricos;
3. preservar os símbolos `x`, `0`, `1`, `2`, `3` e `4` sem interpretação ordinal;
4. manter os 63 registros pertencentes às 31 combinações ambíguas;
5. aplicar one-hot encoding aos sete preditores categóricos, sem padronização;
6. ajustar o encoder somente nos dados de treinamento de cada fold, por meio de pipelines, usando validação cruzada estratificada de cinco folds com embaralhamento e semente 42.
